# xAI Grok 4.3 on Amazon Bedrock Mantle

> **Sample code — not for production.** Provided as AWS Content under the AWS
> Customer Agreement; do not use it in production accounts or on production or other
> critical data. Running these cells calls Amazon Bedrock and incurs charges. Full
> disclaimer in the [README](../README.md#disclaimer).

Grok 4.3 on the `bedrock-mantle` endpoint — a reasoning-first model with
always-on thinking, strong tool use, and a large context window. It is aimed at
document-heavy enterprise work: contract review, case-law research, credit
agreement analysis, financial Q&A.

**Models covered**

| Model ID | Notes |
|---|---|
| `xai.grok-4.3` | Reasoning-first; configurable effort (none/low/medium/high) |

> **Looking for Grok 4.6?** It is a different model with a different endpoint shape
> — `bedrock-runtime` (inference-profile-only) plus `bedrock-mantle` in `us-west-2`
> only, a 500K context window, and an encrypted reasoning trace instead of a
> readable one. See [`02-grok-4-6.ipynb`](02-grok-4-6.ipynb). Grok 4.3, below,
> remains `bedrock-mantle`-only.

## Both OpenAI-compatible APIs, on the `/openai/v1` path
Grok serves **Responses and Chat Completions**, both under `/openai/v1` — the
same prefix as `gemma-4` and `gpt-5.x`, *not* the bare `/v1` used by most
open-weight families.

## Two things to know before you start
1. **Reasoning is always on**, and it spends roughly **400 output tokens before
   any answer text appears**. A tight `max_output_tokens` therefore returns
   HTTP 200 with `status: "incomplete"` and an **empty string**. Budget ≥1000.
2. Its model card documents **non-standard defaults** — `temperature=0.7`,
   `top_p=0.95`, `max_completion_tokens=131072`. For part of 2026 the Responses
   API accepted `temperature` only at that default; it no longer restricts it.
   §3 probes both parameters and derives the verdict rather than asserting one.

## Self-contained, but see also
- **Auth (SigV4 (AWS Signature Version 4) + short-term keys), the three URL paths,
  model discovery** →
  `../00-foundations/01-endpoints-auth-and-the-three-paths.ipynb`
- **Projects, cost attribution, data retention / ZDR (zero data retention),
  CloudWatch** →
  `../00-foundations/02-governance-projects-and-retention.ipynb`
- **Quotas, retry/backoff, service tiers, TTFT (time-to-first-token)** →
  `../00-foundations/03-scaling-tiers-and-latency.ipynb`

## Prerequisites
```bash
pip install -r ../requirements.txt
```

Needs openai, aws-bedrock-token-generator.

`requirements.txt` pins the exact versions this collection was tested
against. An unpinned install resolves whatever is current, which may be
untested or compromised (OWASP LLM03, Supply Chain).

### Where the helpers come from

The next cell does this:

```python
sys.path.insert(0, "../_shared")
from bedrock import ...
```

`bedrock` is **not** a package from PyPI — it is this collection's own helper
module, [`_shared/bedrock.py`](../_shared/bedrock.py). Every notebook sits one
level down, so `../_shared` puts it on the import path. It exists only to remove
repetition; the notebooks are the teaching material, and nothing in the module is
required to call Bedrock yourself.

What this notebook uses from it:

| Helper | What it does |
|---|---|
| `err` | pulls the human-readable message out of an error body, redacted |
| `list_models` | the `bedrock-mantle` model inventory for a Region |
| `parse_json_lenient` | parses the first complete JSON object out of model output, repairing truncated braces |
| `post` | signed JSON HTTP with retries. **Never raises on 4xx/5xx** — it returns `(status, body)` so a cell can *show* an error instead of stopping the notebook |
| `response_text` | assistant text from a Responses API payload |
| `safe_print` | `print()` with account IDs, IAM principals and opaque service IDs redacted |
| `ttft` | times a streaming call: time-to-first-token and output frames/sec |
| `endpoints_for` | answers "mantle, runtime, or both" for a model, from the live catalogues |
| `runtime_models` | the serverless `bedrock-runtime` catalogue with modalities and inference types |

Two behaviours worth knowing before you read any output below, because several
cells depend on them:

- **`post()` and `converse()` never raise on a service error.** They return the
  status and body so a cell can *show* a 400 rather than stopping the notebook.
  Many cells here deliberately provoke an error to demonstrate a limit.
- **Anything printed from a control-plane response goes through redaction**, since
  this output is committed to a public repository.


In [1]:
import json
import sys
import time

sys.path.insert(0, "../_shared")
from bedrock import err, list_models, parse_json_lenient, post, response_text, safe_print, ttft

# Grok is in us-east-1 / us-east-2 / us-west-2 but NOT eu-central-1 (see §9).
REGION = "us-east-1"
GROK = "xai.grok-4.3"

# Same prefix as gemma-4 and gpt-5.x — not the bare /v1.
PREFIX = "/openai/v1"
BASE_URL = f"https://bedrock-mantle.{REGION}.api.aws{PREFIX}"
print("base URL:", BASE_URL)

base URL: https://bedrock-mantle.us-east-1.api.aws/openai/v1


### Which endpoint, and the model ID for each

AWS recommends `bedrock-runtime` for new applications, and since August 2026 it
serves the OpenAI- and Anthropic-compatible APIs as well as Converse. So before
the first call, the question is which endpoint you want — and that has a
complication worth knowing about:

**the same model often carries a different ID on each endpoint.** Send a
`bedrock-mantle` ID to `bedrock-runtime` and you get *"The provided model
identifier is invalid"*, which reads like a missing model rather than a missing
translation.

The cell below asks both catalogues rather than stating an answer that will age.
`runtime_id_for()` returns `None` when a model is genuinely not on
`bedrock-runtime`, which is the honest signal for "you need mantle for this one".

In [2]:
from bedrock import endpoints_for, runtime_id_for

COVERED = [
    "google.gemma-4-31b",
    "xai.grok-4.3",
]

print(f"{'model (as named on mantle)':38} {'on runtime as':40} endpoints")
print("-" * 96)
mantle_only = []
for model_id in COVERED:
    runtime_id = runtime_id_for(model_id, REGION)
    where = endpoints_for(model_id, REGION)
    label = ", ".join(name for name, present in where.items() if present) or "neither"
    if runtime_id is None:
        mantle_only.append(model_id)
    print(f"{model_id:38} {(runtime_id or '-- not on runtime --'):40} {label}")

renamed = [
    m for m in COVERED
    if (r := runtime_id_for(m, REGION)) is not None and r != m
]
# Cross-check the two helpers against each other. A row that prints a runtime id
# next to "mantle" only is self-contradictory, and it happened: endpoints_for()
# compared against a version-stripped catalogue key while runtime_id_for() used the
# full id, so gpt-oss showed a runtime id and "mantle". Neither helper complained.
contradictions = [
    m for m in COVERED
    if (runtime_id_for(m, REGION) is not None)
    != endpoints_for(m, REGION)["runtime"]
]
print()
if contradictions:
    print(f"!! runtime_id_for() and endpoints_for() DISAGREE for {contradictions}.")
    print("   One of them is wrong; do not trust the table above until they agree.")
print(f"=> {len(COVERED) - len(mantle_only)}/{len(COVERED)} of these are on "
      f"bedrock-runtime; {len(renamed)} under a different id.")
if mantle_only:
    print(f"   bedrock-mantle only: {mantle_only}")
    print("   For those, this notebook's endpoint is the only one that serves them.")
else:
    print("   Every model here is on both endpoints. This notebook shows the")
    print("   bedrock-mantle calls; the ids above are what you send to switch.")
print("   Region matters too: a model absent here can be present elsewhere, so")
print("   re-run this in the Region you intend to deploy in.")

model (as named on mantle)             on runtime as                            endpoints
------------------------------------------------------------------------------------------------


google.gemma-4-31b                     -- not on runtime --                     mantle


xai.grok-4.3                           -- not on runtime --                     mantle



=> 0/2 of these are on bedrock-runtime; 0 under a different id.
   bedrock-mantle only: ['google.gemma-4-31b', 'xai.grok-4.3']
   For those, this notebook's endpoint is the only one that serves them.
   Region matters too: a model absent here can be present elsewhere, so
   re-run this in the Region you intend to deploy in.


## 1. First call

Auth is a short-term Bedrock API key minted from ambient IAM credentials —
valid ≤12 h, not refreshable, Region-pinned. (`../00-foundations/01` shows the
self-refreshing provider and the SigV4 alternative that needs no key.)

In [3]:
from aws_bedrock_token_generator import provide_token
from openai import APITimeoutError, OpenAI

# Build from a FRESH token; don't construct once at import and reuse for hours.
# max_retries: bedrock-mantle returns transient 500/503 under load and the SDK
# does NOT retry by default. The `post()` helper used elsewhere already retries.
client = OpenAI(
    api_key=provide_token(region=REGION),
    base_url=BASE_URL,
    max_retries=5,
    timeout=180.0,
)

resp = client.responses.create(
    model=GROK,
    input="Explain what a reasoning-first model is, in two sentences.",
    max_output_tokens=1000,  # >=1000: reasoning spends ~400 before any text
)
print(resp.output_text)
print("\nusage:", resp.usage.model_dump_json())

A reasoning-first model is an AI system (typically an LLM) that generates an explicit chain of intermediate reasoning steps—often internally or via hidden tokens—before producing a final answer or action. This inverts the conventional “answer-first” pattern, allocating additional compute to search, verification, and backtracking so that conclusions are derived rather than merely recalled.

Final answer: reasoning-first models prioritize explicit step-by-step deduction over direct pattern completion, using extra inference-time compute for verification and search before outputting an answer.

usage: {"input_tokens":42,"input_tokens_details":{"cache_write_tokens":0,"cached_tokens":0},"output_tokens":585,"output_tokens_details":{"reasoning_tokens":476},"total_tokens":627}


## 2. Both APIs work — and both live under `/openai/v1`

Probe every combination so the working set is explicit.

In [4]:
# Grok, like every model on the /openai/v1 prefix, takes
# max_completion_tokens rather than max_tokens -- using max_tokens here
# would return a 400 about the parameter and hide the routing lesson.
# One of these combinations does not return a clean 400 but simply STALLS (see the
# note below), so keep attempts=1 and always set a timeout. The timeout has to be
# generous enough for a real answer, though: at 30s this probe reported the
# WORKING prefix as "stalled", because a reasoning-first model can spend that long
# before its first token.
print(f"{'API':22} {'/openai/v1':>14} {'/v1':>18}")
print("-" * 58)
for label, suffix, body in [
    (
        "Responses",
        "/responses",
        {"model": GROK, "input": "Reply OK", "max_output_tokens": 16},
    ),
    (
        "Chat Completions",
        "/chat/completions",
        {
            "model": GROK,
            "messages": [{"role": "user", "content": "Reply OK"}],
            "max_completion_tokens": 16,
        },
    ),
]:
    cells = []
    for prefix in ("/openai/v1", "/v1"):
        started = time.perf_counter()
        code, data = post(
            prefix + suffix, body, region=REGION, attempts=1, timeout=120
        )  # no retries; bounded, but long enough for a reasoning-first model
        elapsed = time.perf_counter() - started
        cells.append(
            f"{code} ({elapsed:.0f}s)" if code != -1 else f"stalled ({elapsed:.0f}s)"
        )
    print(f"{label:22} {cells[0]:>14} {cells[1]:>18}")

API                        /openai/v1                /v1
----------------------------------------------------------


Responses                    200 (2s)     stalled (121s)


Chat Completions             200 (2s)           400 (1s)


### Two lessons here

1. Grok is served on **`/openai/v1`** for both APIs. The bare `/v1` does not work.
2. `/v1/chat/completions` returns a clean, fast **400** ("isn't supported on this
   route"), but **`/v1/responses` does not respond at all — it stalls**. A wrong
   path is therefore not always a fast failure.
3. The timeout you choose decides what you *conclude*. Too short and a slow but
   healthy call is indistinguishable from a stall — which is how the earlier
   version of this table reported the working prefix as stalled.

Practical consequence: **always set a client-side timeout**, and do not retry
indefinitely on a stall. A retry loop with a 240 s timeout and 5 attempts turns
one bad path into a 20-minute hang.

In [5]:
# The clean 400 from the wrong route, for reference.
code, data = post(
    "/v1/chat/completions",
    {
        "model": GROK,
        "messages": [{"role": "user", "content": "Reply OK"}],
        "max_completion_tokens": 16,
    },
    region=REGION,
    attempts=1,
    timeout=30,
)
print(f"/v1/chat/completions -> HTTP {code}: {err(data)[:100]}")

/v1/chat/completions -> HTTP 400: model `xai.grok-4.3` isn't supported on this route


Both APIs are available on `/openai/v1` and both 400 on the bare `/v1`. Prefer
**Responses**: it is the only surface that returns reasoning content, and Grok is
a reasoning-first model, so that matters more here than usual.

## 3. Sampling — probe it, because this surface has moved

Grok's model card documents non-standard defaults (`temperature=0.7`,
`top_p=0.95`), and for a period on `bedrock-mantle` the Responses API accepted
`temperature` **only** at that documented default and refused every other value.
That is no longer the case.

Rather than print today's answer as a rule, sweep both parameters on both models
and let the cell say what it found:

In [6]:
print(f"{'model':24} {'temp=0.7':>10} {'temp=1.0':>10} {'top_p=0.95':>12}")
print("-" * 60)
for model in (GROK, "google.gemma-4-31b"):
    results = []
    for extra in ({"temperature": 0.7}, {"temperature": 1.0}, {"top_p": 0.95}):
        code, data = post(
            f"{PREFIX}/responses",
            {"model": model, "input": "Reply OK", "max_output_tokens": 16, **extra},
            region=REGION,
        )
        results.append("ok" if code == 200 else f"{code}")
    print(f"{model:24} {results[0]:>10} {results[1]:>10} {results[2]:>12}")

model                      temp=0.7   temp=1.0   top_p=0.95
------------------------------------------------------------


xai.grok-4.3                     ok         ok           ok


google.gemma-4-31b               ok         ok           ok


### Read the derived verdict above, not a remembered rule

There was a real pattern here once — each model accepting `temperature` only at
its own documented default, which is why Grok and Gemma 4 disagreed in opposite
directions. It has not survived, and a notebook that wrote it down as a rule would
now be teaching a 400 that no longer happens.

What is durable:

- **Sampling support is per model and per date.** Gate it, and re-probe.
- **Omitting `temperature` and `top_p` always works.** It is the portable default
  and it cannot break when the accepted set changes.
- The **GPT-5.5 and GPT-5.6 families** are the ones that still refuse both today
  (`temperature` at `1.0` only, `top_p` outright), and newer Claude models reject
  both as deprecated. See `../99-cross-cutting/01` for the live matrix.

In [7]:
# Sweep the whole range and DERIVE the conclusion. The previous version of this
# cell printed "only 0.7 is accepted" beneath a table showing every value at 200.
print(f"{'temperature':>12} {'HTTP':>6}  detail")
print("-" * 70)
accepted, refused = [], []
for value in (0.0, 0.5, 0.7, 1.0, 1.5):
    code, data = post(
        f"{PREFIX}/responses",
        {
            "model": GROK,
            "input": "Reply OK",
            "max_output_tokens": 16,
            "temperature": value,
        },
        region=REGION,
    )
    (accepted if code == 200 else refused).append(value)
    print(f"{value:>12} {code:>6}  {'' if code == 200 else err(data)[:48]}")

print()
if not refused:
    print(f"=> every value accepted {accepted}: temperature is unrestricted here today.")
elif accepted == [0.7]:
    print("=> only 0.7 (Grok's documented default) accepted — you are echoing the")
    print("   default back, not tuning anything.")
else:
    print(f"=> accepted {accepted}, refused {refused}")
print("   Either way, omitting temperature is the portable choice.")

 temperature   HTTP  detail
----------------------------------------------------------------------


         0.0    200  


         0.5    200  


         0.7    200  


         1.0    200  


         1.5    200  

=> every value accepted [0.0, 0.5, 0.7, 1.0, 1.5]: temperature is unrestricted here today.
   Either way, omitting temperature is the portable choice.


In [8]:
# max_output_tokens minimum is 16 on the Responses API.
for n in (8, 16):
    code, data = post(
        f"{PREFIX}/responses",
        {"model": GROK, "input": "Hi", "max_output_tokens": n},
        region=REGION,
    )
    detail = "" if code == 200 else err(data)[:70]
    print(f"max_output_tokens={n:3} -> HTTP {code} {detail}")

max_output_tokens=  8 -> HTTP 400 Invalid 'max_output_tokens': integer below minimum value. Expected a v


max_output_tokens= 16 -> HTTP 200 


## 4. Reasoning — always on, effort configurable

Grok reasons on every request rather than treating thinking as optional. That
makes it behave consistently across multi-step agent loops, at the cost of
spending reasoning tokens even on easy questions.

In [9]:
for effort in ("none", "minimal", "low", "medium", "high"):
    code, data = post(
        f"{PREFIX}/responses",
        {
            "model": GROK,
            "input": "2+2?",
            "max_output_tokens": 32,
            "reasoning": {"effort": effort},
        },
        region=REGION,
    )
    print(f"  effort={effort:8} -> HTTP {code} {'' if code == 200 else err(data)[:60]}")

  effort=none     -> HTTP 200 


  effort=minimal  -> HTTP 400 Unsupported value: 'minimal' is not supported with the 'xai.


  effort=low      -> HTTP 200 


  effort=medium   -> HTTP 200 


  effort=high     -> HTTP 200 


In [10]:
# Each row is a SINGLE sample, so read the ordering, not the absolute numbers -
# Grok's wall-clock time swings widely with endpoint load.
print(f"{'effort':8} {'reasoning tok':>14} {'output tok':>11} {'latency':>9}")
print("-" * 46)
for effort in ("none", "low", "medium"):
    started = time.perf_counter()
    try:
        r = client.with_options(timeout=180.0).responses.create(
            model=GROK,
            input="What is 47 * 89? Answer with the number only.",
            reasoning={"effort": effort},
            max_output_tokens=1500,
        )
    except APITimeoutError:
        # A timeout here is a load signal, not a bug. Report and keep the table
        # going rather than aborting the notebook.
        print(f"{effort:8} {'timed out after 180s (endpoint under load)':>38}")
        continue
    elapsed = time.perf_counter() - started
    print(
        f"{effort:8} {r.usage.output_tokens_details.reasoning_tokens:>14} "
        f"{r.usage.output_tokens:>11} {elapsed:>8.2f}s"
    )
print()
print("Reasoning tokens rise with effort. Wall-clock time does NOT track it")
print("reliably - queue time dominates, so a 'medium' call can finish before a")
print("'low' one. Read the token column for cost and treat latency as noisy.")
print("'high' is omitted here: it can exceed 15 minutes for one call under load.")

effort    reasoning tok  output tok   latency
----------------------------------------------


none                  0           7     1.57s


low                 314         433     1.94s


medium              532         659     3.23s

Reasoning tokens rise with effort. Wall-clock time does NOT track it
reliably - queue time dominates, so a 'medium' call can finish before a
'low' one. Read the token column for cost and treat latency as noisy.
'high' is omitted here: it can exceed 15 minutes for one call under load.


### Set a client timeout on reasoning calls

Grok's wall-clock time varies a lot with load: the same request has taken 45s and
well over 15 minutes on different days. A reasoning call with no client timeout is
therefore a latent hang in your application, not just a slow cell.

The SDK's default timeout is generous, so set your own — and pick `effort` for the
job rather than reaching for `high` by default. `medium` answers this question just
as well and costs a fraction of the wait.

In [11]:
# Escalate the budget until the answer appears. Grok spends its reasoning tokens
# FIRST, so an under-budgeted call returns HTTP 200, status="incomplete", and an
# empty string -- reasoning consumed everything before any answer text.
#
# `timeout=` matters as much as the budget: bound the wait explicitly, because a
# reasoning call has no natural upper bound on latency.
grok = client.with_options(timeout=240.0)
QUESTION = (
    "A contract says payment is due 'within 30 days of invoice receipt, "
    "excluding public holidays'. The invoice arrives 1 December. Name two "
    "ambiguities a reviewer should flag."
)

print(f"{'budget':>8} {'status':>12} {'reasoning tok':>14} {'answer chars':>13}")
print("-" * 52)
reasoned = None
for budget in (1200, 2500, 4000):
    attempt = grok.responses.create(
        model=GROK,
        input=QUESTION,
        reasoning={"effort": "medium"},  # 'high' can take many minutes under load
        max_output_tokens=budget,
    )
    used = attempt.usage.output_tokens_details.reasoning_tokens
    text = attempt.output_text or ""
    print(f"{budget:>8} {attempt.status:>12} {used:>14} {len(text):>13}")
    if text.strip():
        reasoned = attempt
        break

if reasoned is None:
    raise RuntimeError(
        "no answer text even at 4000 tokens - reasoning consumed the whole budget"
    )

print("\noutput item types:", [i.type for i in reasoned.output])
print("=== ANSWER ===")
print(reasoned.output_text[:500])

  budget       status  reasoning tok  answer chars
----------------------------------------------------


    1200   incomplete           1197             0


    2500    completed           1026           819

output item types: ['reasoning', 'message']
=== ANSWER ===
Two key ambiguities stand out for a reviewer:

- "Excluding public holidays" does not clarify the counting rule: whether the 30-day window is lengthened by the number of holidays that fall inside it, or whether the deadline is simply postponed if it lands on a holiday (or whether only non-holiday days are counted at all).
- The clause gives no indication of whose or which public holidays apply (e.g., the supplier’s jurisdiction, the customer’s headquarters, the governing-law country, or all loca


## 4b. Budget for reasoning, or you get an empty answer

This is the single most important operational fact about Grok. Because reasoning
always runs first, a small `max_output_tokens` is consumed entirely by thinking.
The request succeeds — **HTTP 200** — but `status` is `incomplete` and the text is
empty. Sweep the budget and watch where text starts appearing:

In [12]:
print(f"{'max_output_tokens':>18} {'status':>12} {'reason tok':>11} {'chars':>7}")
print("-" * 54)
for budget in (400, 1000, 2000):
    code, data = post(
        f"{PREFIX}/responses",
        {
            "model": GROK,
            "input": "List three contract risks.",
            "reasoning": {"effort": "low"},
            "max_output_tokens": budget,
        },
        region=REGION,
    )
    details = (data.get("usage") or {}).get("output_tokens_details") or {}
    text = response_text(data)
    print(
        f"{budget:>18} {str(data.get('status')):>12} "
        f"{details.get('reasoning_tokens'):>11} {len(text):>7}"
    )

 max_output_tokens       status  reason tok   chars
------------------------------------------------------


               400   incomplete         397       0


              1000    completed         522     697


              2000    completed         443     288


Read the `chars` column. Where it is `0`, reasoning consumed the entire budget and
the call returned HTTP 200 with an empty string; the exact budget at which that
happens moves run to run, because how long the model thinks is not fixed. §7's
sweep (`max_output_tokens=400`) has returned `0` chars; this one sometimes returns
a short answer at the same budget. That variability *is* the lesson — you cannot
pick a budget once and assume it is safe. The practical rules:

- **Always check `status`** (or that the text is non-empty) before using a result.
  Never infer success from HTTP 200 on this model.
- Budget **≥1000** output tokens for anything you expect an answer from, and treat
  an empty answer as a retry-with-more-budget path rather than an error.
- `reasoning={"effort": "none"}` disables thinking, which frees the whole budget
  for the answer — useful for simple extraction where you do not need reasoning.

In [13]:
for effort in ("low", "none"):
    code, data = post(
        f"{PREFIX}/responses",
        {
            "model": GROK,
            "input": "List three contract risks.",
            "reasoning": {"effort": effort},
            "max_output_tokens": 1000,
        },
        region=REGION,
    )
    details = (data.get("usage") or {}).get("output_tokens_details") or {}
    print(
        f"effort={effort:5} reasoning_tokens={details.get('reasoning_tokens'):>4} "
        f"-> {response_text(data)[:70]!r}"
    )

effort=low   reasoning_tokens= 372 -> '1. Non-performance (e.g., missed deliverables or deadlines)  \n2. Ambig'


effort=none  reasoning_tokens=   0 -> '1. Price-escalation risk – the possibility that input costs, labor rat'


## 4c. Reasoning content is encrypted

Unlike Gemma 4 or gpt-5.x, Grok does not return readable reasoning text. Its model
card documents that the trace is **encrypted**, retrievable with
`include: ["reasoning.encrypted_content"]`, and can be passed back on later turns
to carry reasoning context forward.

In [14]:
code, data = post(
    f"{PREFIX}/responses",
    {
        "model": GROK,
        "input": "Why is the sky blue? One sentence.",
        "reasoning": {"effort": "low"},
        "include": ["reasoning.encrypted_content"],
        "max_output_tokens": 1000,
    },
    region=REGION,
)
print("HTTP", code, "| items:", [i.get("type") for i in data.get("output", [])])
for item in data.get("output", []):
    if item.get("type") == "reasoning":
        encrypted = item.get("encrypted_content")
        print(f"   reasoning item keys : {sorted(item.keys())}")
        print(
            f"   encrypted_content   : {'present' if encrypted else 'absent'}"
            f"{f' ({len(encrypted)} chars)' if encrypted else ''}"
        )
print("answer:", response_text(data)[:150])

HTTP 200 | items: ['reasoning', 'message']
   reasoning item keys : ['content', 'encrypted_content', 'id', 'summary', 'type']
   encrypted_content   : present (1324 chars)
answer: The sky appears blue due to Rayleigh scattering, whereby shorter (blue) wavelengths of sunlight are scattered far more than longer ones by atmospheric


## 5. Streaming

Reasoning and answer text arrive as separate event types.

In [15]:
stream = client.responses.create(
    model=GROK,
    input="List three risks in a credit agreement review.",
    reasoning={"effort": "low"},
    max_output_tokens=1200,
    stream=True,
)
counts = {}
print("--- live ---")
for event in stream:
    counts[event.type] = counts.get(event.type, 0) + 1
    if event.type == "response.reasoning_text.delta":
        print("\033[2m" + event.delta + "\033[0m", end="", flush=True)
    elif event.type == "response.output_text.delta":
        print(event.delta, end="", flush=True)
print("\n\n--- event types ---")
for name, count in sorted(counts.items(), key=lambda kv: -kv[1]):
    print(f"  {count:4}  {name}")

--- live ---


**

Three

 key

 risks

...

 fees

/

acceleration

 risks



--- event types ---
   206  response.output_text.delta
     2  response.output_item.added
     2  response.output_item.done
     1  response.created
     1  response.in_progress
     1  response.content_part.added
     1  response.output_text.done
     1  response.content_part.done
     1  response.completed


## 6. Multi-turn: history array vs server-side state

In [16]:
conversation = [
    {"role": "system", "content": "You are a precise legal-operations assistant."},
    {"role": "user", "content": "What is a force majeure clause?"},
]
first = client.responses.create(model=GROK, input=conversation, max_output_tokens=1000)
print("assistant:", first.output_text[:180])

conversation += [
    {"role": "assistant", "content": first.output_text},
    {"role": "user", "content": "Name one event commonly excluded from it."},
]
second = client.responses.create(model=GROK, input=conversation, max_output_tokens=1000)
print("\nassistant:", second.output_text[:180])

assistant: A **force majeure clause** is a contractual provision that excuses one or both parties from performing their contractual obligations when an unforeseen event beyond their reasonabl



assistant: **Economic hardship or changes in market conditions** (e.g., a downturn that merely makes performance unprofitable or more expensive). Force majeure clauses do not excuse performan


Append only **final answers** to history — not reasoning items. Replaying a
model's own reasoning degrades later turns.

In [17]:
# Server-side state needs store=True, which retains input+output for 30 days
# in-Region (see ../00-foundations/02 for the retention/ZDR controls).
turn1 = client.responses.create(
    model=GROK,
    input="Our governing law is Singapore. Reply: noted.",
    max_output_tokens=1000,
    store=True,
)
turn2 = client.responses.create(
    model=GROK,
    input="Which governing law did I state?",
    previous_response_id=turn1.id,
    max_output_tokens=1000,
)
print("chained recall:", turn2.output_text[:140])

unstored = client.responses.create(
    model=GROK, input="Confidential. Reply ok.", max_output_tokens=1000, store=False
)
code, data = post(
    f"{PREFIX}/responses",
    {
        "model": GROK,
        "input": "What did I say?",
        "previous_response_id": unstored.id,
        "max_output_tokens": 1000,
    },
    region=REGION,
)
print(f"chaining from store=False -> HTTP {code}: {err(data)[:80]}")

chained recall: **Singapore**


chaining from store=False -> HTTP 404: Response not found.


## 7. Tool use and structured output

Grok has strong tool use, which is the main reason to pick it for agent loops.
Note the **flat** Responses tool shape.

In [18]:
def get_filing(company: str, year: int) -> dict:
    """Stand-in for a filings database."""
    table = {("acme", 2025): {"revenue_musd": 812, "net_margin_pct": 11.4}}
    hit = table.get((company.lower(), year))
    return {"company": company, "year": year, "found": bool(hit), **(hit or {})}


filing_tool = {
    "type": "function",
    "name": "get_filing",
    "description": "Fetch headline financials for a company and year.",
    "parameters": {
        "type": "object",
        "properties": {
            "company": {"type": "string"},
            "year": {"type": "integer"},
        },
        "required": ["company", "year"],
    },
}

# NOTE: we use the retrying `post()` helper rather than the bare SDK here.
# bedrock-mantle returns transient 500s under load, and the SDK does not retry
# them — a tool loop that runs several calls will eventually hit one.
convo = [{"role": "user", "content": "What was Acme's 2025 net margin?"}]
code, data = post(
    f"{PREFIX}/responses",
    {
        "model": GROK,
        "input": convo,
        "tools": [filing_tool],
        "tool_choice": "auto",
        "max_output_tokens": 1200,
    },
    region=REGION,
)
print("HTTP", code)
calls = [i for i in data.get("output", []) if i.get("type") == "function_call"]
print("tool calls:", [(c["name"], c["arguments"]) for c in calls])

for call in calls:
    args = parse_json_lenient(call["arguments"])
    result = get_filing(**args)
    convo.append(
        {
            "type": "function_call",
            "call_id": call["call_id"],
            "name": call["name"],
            "arguments": call["arguments"],
        }
    )
    convo.append(
        {
            "type": "function_call_output",
            "call_id": call["call_id"],
            "output": json.dumps(result),
        }
    )

code, final = post(
    f"{PREFIX}/responses",
    {"model": GROK, "input": convo, "tools": [filing_tool], "max_output_tokens": 1000},
    region=REGION,
)
print("\nfinal answer:", response_text(final)[:220])

HTTP 200
tool calls: [('get_filing', '{"company":"Acme","year":2025}')]



final answer: **11.4%** 

The filing data for Acme in 2025 directly reports a net margin of 11.4%.


In [19]:
# Strict JSON via the native schema route.
schema = {
    "type": "object",
    "properties": {
        "clause_type": {"type": "string"},
        "risk": {"type": "string", "enum": ["low", "medium", "high"]},
        "rationale": {"type": "string"},
    },
    "required": ["clause_type", "risk", "rationale"],
    "additionalProperties": False,
}

code, data = post(
    f"{PREFIX}/responses",
    {
        "model": GROK,
        "input": "Classify: 'Either party may terminate immediately without notice.'",
        "max_output_tokens": 1500,
        "text": {
            "format": {
                "type": "json_schema",
                "name": "clause",
                "schema": schema,
                "strict": True,
            }
        },
    },
    region=REGION,
)
raw = response_text(data)
print("HTTP", code, "| raw:", repr(raw[:160]))
# parse_json_lenient rather than json.loads: models can append characters after a
# valid object, and an empty body is possible if the token budget runs out first.
if raw.strip():
    print(json.dumps(parse_json_lenient(raw), indent=2))
else:
    print("empty output — raise max_output_tokens (reasoning consumed the budget)")

HTTP 200 | raw: '{\n  "clause_type": "Termination",\n  "risk": "high",\n  "rationale": "Allows either party to end the contract instantly with no notice or cure period, creating hi'
{
  "clause_type": "Termination",
  "risk": "high",
  "rationale": "Allows either party to end the contract instantly with no notice or cure period, creating high uncertainty and potential for abrupt disruption."
}


**Watch the budget on a reasoning-first model.** Grok thinks before it writes, so
a tight `max_output_tokens` can be exhausted before any JSON is emitted, giving
HTTP 200 with an empty string. Always check for content before parsing.

In [20]:
print(f"{'max_output_tokens':>18} {'chars returned':>16}")
print("-" * 36)
for budget in (400, 1000, 2000):
    code, data = post(
        f"{PREFIX}/responses",
        {
            "model": GROK,
            "input": "Classify: 'Termination without notice.'",
            "max_output_tokens": budget,
            "text": {
                "format": {
                    "type": "json_schema",
                    "name": "clause",
                    "schema": schema,
                    "strict": True,
                }
            },
        },
        region=REGION,
    )
    print(f"{budget:>18} {len(response_text(data)):>16}")

 max_output_tokens   chars returned
------------------------------------


               400                0


              1000              195


              2000              209


## 8. Does the built-in Web Search tool work here?

Server-side tool support is per model and changes, so the cell below asks rather
than assumes. In the run committed here it came back as an explicit 400, which is
useful either way: a clear rejection tells you to plan for your own retrieval step
instead of discovering the gap later.
(See `../01-openai-gpt/02-web-search-and-grounding.ipynb` for the tool in use.)

In [21]:
code, data = post(
    f"{PREFIX}/responses",
    {
        "model": GROK,
        "input": "What happened in the news today?",
        "max_output_tokens": 1000,
        "tools": [{"type": "web_search"}],
    },
    region=REGION,
)
print(f"web_search on Grok -> HTTP {code}")
print("message:", err(data)[:130])

web_search on Grok -> HTTP 400
message: Tool type 'web_search' is not supported for model `xai.grok-4.3`.


## 9. Regional availability

In [22]:
regions = ("us-east-1", "us-east-2", "us-west-2", "eu-central-1")
print(f"{'region':14} {'grok-4.3':>10}")
print("-" * 26)
for region in regions:
    try:
        present = GROK in list_models(region)
    except (RuntimeError, OSError) as exc:
        print(f"{region:14} {type(exc).__name__}")
        continue
    print(f"{region:14} {('yes' if present else '-'):>10}")

region           grok-4.3
--------------------------


us-east-1             yes


us-east-2             yes


us-west-2             yes


eu-central-1            -


## 10. Latency and service tiers

In [23]:
print(f"{'tier':10} {'TTFT (s)':>10} {'total (s)':>10} {'frames/s':>10}")
print("-" * 44)
for tier in ("default", "flex", "priority"):
    m = ttft(
        f"{PREFIX}/responses",
        {
            "model": GROK,
            "input": "List three contract review checkpoints.",
            "max_output_tokens": 1000,
            "service_tier": tier,
        },
        region=REGION,
    )
    if m.get("error"):
        # Do NOT label every failure "tier not supported". A URLError or a timeout
        # is a transport problem and says nothing about whether the parameter is
        # accepted; reporting it as an unsupported feature invents a limitation
        # the service never claimed. Read the error before attributing a cause.
        detail = str(m["error"])
        if "service_tier" in detail or "unsupported" in detail.lower():
            cause = "tier refused by this model"
        else:
            cause = "transport error - retry; tells you nothing about tier support"
        print(f"{tier:10} {detail[:30]:>32}  ({cause})")
    else:
        print(
            f"{tier:10} {m['ttft_s']:>10.3f} {m['total_s']:>10.3f} "
            f"{m['frames_per_s']:>10.1f}"
        )

tier         TTFT (s)  total (s)   frames/s
--------------------------------------------


default         0.818      3.956       64.7


flex            0.807      4.213       63.7


priority        0.790      4.114       67.7


Read this as one sample, not a benchmark. `flex` is deliberately deprioritised, so
its TTFT is usually the worst of the three; `default` and `priority` are close on an
idle account and separate under contention. A single `priority` call has also taken
**54 seconds** here while `default` took 4 — queue placement is not a guarantee.
Measure over many calls before quoting a number. See `../00-foundations/03`.

## 11. A worked enterprise example

The workload Grok is positioned for: read a document, extract structured
findings, flag risk.

In [24]:
CONTRACT = """
SERVICES AGREEMENT (extract)
4.1 The Supplier shall deliver the Services with reasonable skill and care.
4.2 Either party may terminate this Agreement immediately upon written notice
    if the other party commits a material breach that remains unremedied for
    fourteen (14) days.
7.3 The Supplier's total liability shall not exceed the fees paid in the three
    (3) months preceding the claim.
9.1 This Agreement is governed by the laws of Singapore.
"""

findings_schema = {
    "type": "object",
    "properties": {
        "governing_law": {"type": "string"},
        "liability_cap": {"type": "string"},
        "termination_notice_days": {"type": "integer"},
        "concerns": {"type": "array", "items": {"type": "string"}},
        "overall_risk": {"type": "string", "enum": ["low", "medium", "high"]},
    },
    "required": [
        "governing_law",
        "liability_cap",
        "termination_notice_days",
        "concerns",
        "overall_risk",
    ],
    "additionalProperties": False,
}

code, data = post(
    f"{PREFIX}/responses",
    {
        "model": GROK,
        "input": [
            {
                "role": "system",
                "content": "You are a contract reviewer. Be conservative.",
            },
            {
                "role": "user",
                "content": f"Review this extract and return findings.\n{CONTRACT}",
            },
        ],
        "reasoning": {"effort": "medium"},
        "max_output_tokens": 2000,  # generous: reasoning runs before the JSON
        "text": {
            "format": {
                "type": "json_schema",
                "name": "findings",
                "schema": findings_schema,
                "strict": True,
            }
        },
        "store": False,
    },
    region=REGION,
)
raw = response_text(data)
print("HTTP", code, "| chars:", len(raw))
if raw.strip():
    print(json.dumps(parse_json_lenient(raw), indent=2))
else:
    print("empty — raise max_output_tokens further")

HTTP 200 | chars: 570
{
  "governing_law": "Singapore",
  "liability_cap": "fees paid in the three (3) months preceding the claim",
  "termination_notice_days": 14,
  "concerns": [
    "Liability cap limited to preceding 3 months' fees may be insufficient to cover material losses or damages",
    "Only 14-day cure period before termination for material breach; no termination for convenience or other termination rights specified",
    "Liability limitation applies only to Supplier, creating potential asymmetry and unlimited exposure for the other party"
  ],
  "overall_risk": "medium"
}


## 12. Production hardening

In [25]:
code, project = post(
    "/v1/organization/projects",
    {
        "name": "grok-samples",
        "tags": {"Application": "GrokDemo", "Environment": "Demo"},
    },
    region=REGION,
)
project_id = project.get("id")
safe_print("project:", code, project_id)


class GrokClient:
    """Production shape: no temperature, generous budget, retries, attribution."""

    def __init__(self, region=REGION, tier="default", project=None):
        self.model, self.region, self.tier, self.project = GROK, region, tier, project

    def ask(self, prompt, *, effort="low", max_output_tokens=1500, schema=None):
        body = {
            "model": self.model,
            "input": prompt,
            # Reasoning-first: budget generously or you get an empty answer.
            # Floor of 1000: reasoning spends ~400 tokens before any text.
            "max_output_tokens": max(1000, max_output_tokens),
            "reasoning": {"effort": effort},
            "service_tier": self.tier,
            "store": False,  # opt out of the 30-day retention default
            # temperature deliberately omitted: rejected by this model.
            "top_p": 0.95,  # Grok's documented default; accepted
        }
        if schema:
            body["text"] = {
                "format": {
                    "type": "json_schema",
                    "name": "out",
                    "schema": schema,
                    "strict": True,
                }
            }
        headers = {"OpenAI-Project": self.project} if self.project else None
        # post() retries 429/5xx with exponential backoff.
        code, data = post(
            f"{PREFIX}/responses", body, region=self.region, headers=headers
        )
        if code != 200:
            raise RuntimeError(f"HTTP {code}: {err(data)}")
        text = response_text(data)
        if schema:
            if not text.strip():
                raise RuntimeError("empty output — raise max_output_tokens")
            return parse_json_lenient(text)
        return text


grok = GrokClient(project=project_id)

print("plain      :", grok.ask("Name one benefit of always-on reasoning.")[:140])

RISK_SCHEMA = {
    "type": "object",
    "properties": {"risk": {"type": "string", "enum": ["low", "medium", "high"]}},
    "required": ["risk"],
    "additionalProperties": False,
}
print(
    "structured :",
    grok.ask(
        "Classify the risk of: 'Unlimited liability for the supplier.'",
        schema=RISK_SCHEMA,
        max_output_tokens=1500,
    ),
)

project: 200 proj_s7yvrihm...


plain      : Proactive (vs. reactive) behavior / real-time adaptation to change 

The core idea behind always-on (continuous) reasoning is that inference


structured : {'risk': 'high'}


In [26]:
code, archived = post(
    f"/v1/organization/projects/{project_id}/archive", {}, region=REGION
)
print("archived:", code, archived.get("status"))

archived: 200 archived


## Gotchas — Grok 4.3 on bedrock-mantle

| Gotcha | Detail |
|---|---|
| Sampling surface **moves** | `temperature` was once accepted only at Grok's documented `0.7`; §3 probes it and derives the verdict. Omitting it always works |
| Documented defaults ≠ constraints | The model card's `temperature=0.7` is a default, not necessarily the only accepted value |
| Path prefix | `/openai/v1`. Bare `/v1` chat/completions 400s fast, but **`/v1/responses` stalls** |
| Always set timeouts | A stalling path plus a retry loop = a multi-minute hang |
| Transient 500s | The service returns them under load; the SDK does **not** retry — wrap your calls |
| Reasoning always on | Spends ~400 tokens first — budget **≥1000** or you get empty output |
| `status: "incomplete"` | HTTP 200 + empty string when reasoning eats the budget — check `status` |
| Reasoning is encrypted | Not human-readable; use `include:["reasoning.encrypted_content"]` |
| Documented defaults | `temperature=0.7`, `top_p=0.95`, `max_completion_tokens=131072` |
| `max_output_tokens` | Minimum **16** |
| `reasoning.effort` | `none`/`low`/`medium`/`high`; **`minimal` rejected** |
| `web_search` | Probe it (§8) — server-side tool support is per model and changes |
| `store=False` | Blocks `previous_response_id` chaining (404) |
| Region | Absent from eu-central-1 |
| Probe timeouts | Too short a timeout turns a slow healthy call into a false "stall" (§2) |

## Where next
- Same path prefix: `../03-google-gemma/`, `../01-openai-gpt/`
- Web Search: `../01-openai-gpt/02-web-search-and-grounding.ipynb`
- Different API shape: `../04-qwen/` (Chat Completions), `../02-anthropic-claude/`
  (Messages)

## Also on `bedrock-runtime`? Grok — no

Grok 4.3 is **`bedrock-mantle` only**, and only on the `/openai/v1` prefix. There is no Converse path, so the endpoint choice is made for you.

The cell below confirms it against the live catalogues rather than asserting it,
because model availability moves.


In [27]:
from bedrock import endpoints_for, runtime_models

MODEL = "xai.grok-4.3"
where = endpoints_for(MODEL)
print(f"{MODEL} -> {where}")

if not where["runtime"]:
    print("\nNot on bedrock-runtime, so Converse is not an option for this model.")
    print("Same-provider models that ARE on bedrock-runtime today:")
    provider = MODEL.split(".")[0]
    siblings = sorted(m for m in runtime_models() if m.startswith(provider + "."))
    for sibling in siblings[:8]:
        print("   ", sibling)
    if not siblings:
        print("    (none)")


xai.grok-4.3 -> {'mantle': True, 'runtime': False}

Not on bedrock-runtime, so Converse is not an option for this model.
Same-provider models that ARE on bedrock-runtime today:
    xai.grok-4.6
